In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

In [2]:
# Paths

DATA_DIR = Path("../outputs/candidate_features")
OUTPUT_DIR = Path("../outputs/regularisation_abs_logreturn")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Data directory: C:\Users\franc\OneDrive\Geop-Model\outputs\candidate_features
Output directory: C:\Users\franc\OneDrive\Geop-Model\outputs\regularisation_abs_logreturn


In [3]:
# Load candidate-feature datasets

deepseek = pd.read_csv(
    DATA_DIR / "candidate_features_deepseek.csv",
    parse_dates=["observation_date"]
)

gemma = pd.read_csv(
    DATA_DIR / "candidate_features_gemma.csv",
    parse_dates=["observation_date"]
)

manifest = pd.read_csv(
    DATA_DIR / "candidate_feature_manifest.csv"
)

print("DeepSeek:", deepseek.shape)
print("Gemma:", gemma.shape)

print(
    "DeepSeek sample:",
    deepseek["observation_date"].min(),
    "to",
    deepseek["observation_date"].max()
)

print(
    "Gemma sample:",
    gemma["observation_date"].min(),
    "to",
    gemma["observation_date"].max()
)

DeepSeek: (982, 126)
Gemma: (982, 126)
DeepSeek sample: 2017-01-17 00:00:00 to 2021-01-08 00:00:00
Gemma sample: 2017-01-17 00:00:00 to 2021-01-08 00:00:00


In [31]:
# Verify chronological ordering before constructing t+1 target

for name, df in {
    "DeepSeek": deepseek,
    "Gemma": gemma
}.items():

    df.sort_values("observation_date", inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(
        f"{name} dates sorted:",
        df["observation_date"].is_monotonic_increasing
    )

    print(
        f"{name} duplicate dates:",
        df["observation_date"].duplicated().sum()
    )

DeepSeek dates sorted: True
DeepSeek duplicate dates: 0
Gemma dates sorted: True
Gemma duplicate dates: 0


In [4]:
# Absolute NEXT-DAY EUR/USD log return as volatility proxy

TARGET_RETURN = "DEXUSEU_logreturn"
TARGET_VOL = "DEXUSEU_abs_logreturn_tplus1"

for df in [deepseek, gemma]:
    # Predictors at time t -> absolute EUR/USD log return at t+1
    df[TARGET_VOL] = df[TARGET_RETURN].shift(-1).abs()

print("Target:", TARGET_VOL)

print("\nDeepSeek:")
display(deepseek[[TARGET_RETURN, TARGET_VOL]].describe())

print("\nGemma:")
display(gemma[[TARGET_RETURN, TARGET_VOL]].describe())

Target: DEXUSEU_abs_logreturn_tplus1

DeepSeek:


C:\Users\franc\AppData\Local\Temp\ipykernel_26964\2445974320.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TARGET_VOL] = df[TARGET_RETURN].shift(-1).abs()


,DEXUSEU_logreturn,DEXUSEU_abs_logreturn_tplus1
count,982.000000,981.000000
mean,0.000145,0.003142
std,0.004092,0.002619
min,-0.017799,0.000000
25%,-0.002313,0.001149
50%,0.000086,0.002525
75%,0.002721,0.004475
max,0.017384,0.017799



Gemma:


,DEXUSEU_logreturn,DEXUSEU_abs_logreturn_tplus1
count,982.000000,981.000000
mean,0.000145,0.003142
std,0.004092,0.002619
min,-0.017799,0.000000
25%,-0.002313,0.001149
50%,0.000086,0.002525
75%,0.002721,0.004475
max,0.017384,0.017799


In [5]:
# Define predictor columns
# Exclude date, original signed return, and new absolute-return target

EXCLUDE_COLS = {
    "observation_date",
    TARGET_RETURN,
    TARGET_VOL,
}

deepseek_predictors = [
    col for col in deepseek.columns
    if col not in EXCLUDE_COLS
]

gemma_predictors = [
    col for col in gemma.columns
    if col not in EXCLUDE_COLS
]

print("DeepSeek predictors:", len(deepseek_predictors))
print("Gemma predictors:", len(gemma_predictors))

print(
    "Same predictor count:",
    len(deepseek_predictors) == len(gemma_predictors)
)

print(
    "Signed return excluded:",
    TARGET_RETURN not in deepseek_predictors
)

print(
    "Abs-return target excluded:",
    TARGET_VOL not in deepseek_predictors
)

DeepSeek predictors: 124
Gemma predictors: 124
Same predictor count: True
Signed return excluded: True
Abs-return target excluded: True


In [6]:
# Build aligned X and y for next-day volatility target

valid_ds = deepseek[TARGET_VOL].notna()
valid_gm = gemma[TARGET_VOL].notna()

X_deepseek = deepseek.loc[valid_ds, deepseek_predictors].copy()
X_gemma = gemma.loc[valid_gm, gemma_predictors].copy()

y_deepseek = deepseek.loc[valid_ds, TARGET_VOL].copy()
y_gemma = gemma.loc[valid_gm, TARGET_VOL].copy()

print("DeepSeek X:", X_deepseek.shape)
print("Gemma X:", X_gemma.shape)

print("DeepSeek y:", y_deepseek.shape)
print("Gemma y:", y_gemma.shape)

print("\nMissing values:")
print("DeepSeek X:", X_deepseek.isna().sum().sum())
print("Gemma X:", X_gemma.isna().sum().sum())
print("DeepSeek y:", y_deepseek.isna().sum())
print("Gemma y:", y_gemma.isna().sum())

DeepSeek X: (981, 124)
Gemma X: (981, 124)
DeepSeek y: (981,)
Gemma y: (981,)

Missing values:
DeepSeek X: 0
Gemma X: 0
DeepSeek y: 0
Gemma y: 0


In [7]:
# Chronological 80/20 train-test split

n = len(X_deepseek)
split_idx = int(n * 0.80)

# DeepSeek
X_train_ds = X_deepseek.iloc[:split_idx].copy()
X_test_ds = X_deepseek.iloc[split_idx:].copy()

y_train_ds = y_deepseek.iloc[:split_idx].copy()
y_test_ds = y_deepseek.iloc[split_idx:].copy()

# Gemma
X_train_gm = X_gemma.iloc[:split_idx].copy()
X_test_gm = X_gemma.iloc[split_idx:].copy()

y_train_gm = y_gemma.iloc[:split_idx].copy()
y_test_gm = y_gemma.iloc[split_idx:].copy()

# Dates for checking — predictor dates at time t
valid_dates = deepseek.loc[valid_ds, "observation_date"].reset_index(drop=True)

train_dates = valid_dates.iloc[:split_idx]
test_dates = valid_dates.iloc[split_idx:]

print("Split index:", split_idx)

print("\nTRAIN")
print("Observations:", len(X_train_ds))
print("Dates:", train_dates.min(), "to", train_dates.max())

print("\nTEST")
print("Observations:", len(X_test_ds))
print("Dates:", test_dates.min(), "to", test_dates.max())

print("\nShapes:")
print("DeepSeek:", X_train_ds.shape, X_test_ds.shape)
print("Gemma:   ", X_train_gm.shape, X_test_gm.shape)

Split index: 784

TRAIN
Observations: 784
Dates: 2017-01-17 00:00:00 to 2020-03-20 00:00:00

TEST
Observations: 197
Dates: 2020-03-23 00:00:00 to 2021-01-06 00:00:00

Shapes:
DeepSeek: (784, 124) (197, 124)
Gemma:    (784, 124) (197, 124)


In [8]:
# Time-series cross-validation

N_SPLITS = 5

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

# Same alpha grid as primary regularisation experiment
alphas = np.logspace(-6, 0, 200)

print("CV folds:", N_SPLITS)
print("Number of alpha values:", len(alphas))
print("Alpha range:", alphas.min(), "to", alphas.max())

print("\nDeepSeek training fold sizes:")
for i, (train_idx, val_idx) in enumerate(tscv.split(X_train_ds), start=1):
    print(
        f"Fold {i}:",
        f"train={len(train_idx)},",
        f"validation={len(val_idx)}"
    )

CV folds: 5
Number of alpha values: 200
Alpha range: 1e-06 to 1.0

DeepSeek training fold sizes:
Fold 1: train=134, validation=130
Fold 2: train=264, validation=130
Fold 3: train=394, validation=130
Fold 4: train=524, validation=130
Fold 5: train=654, validation=130


In [9]:
# Leakage-safe LASSO:
# scaling is fitted separately inside each CV training fold

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=100000))
])

lasso_param_grid = {
    "lasso__alpha": alphas
}

lasso_cv_ds = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

lasso_cv_gm = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# IMPORTANT: use UNSCALED X here
lasso_cv_ds.fit(X_train_ds, y_train_ds)
lasso_cv_gm.fit(X_train_gm, y_train_gm)

print(
    "DeepSeek best alpha:",
    lasso_cv_ds.best_params_["lasso__alpha"]
)

print(
    "Gemma best alpha:",
    lasso_cv_gm.best_params_["lasso__alpha"]
)

DeepSeek best alpha: 0.00039171014908092564
Gemma best alpha: 0.00048241087041653687


In [10]:
# Extract non-zero coefficients from leakage-safe LASSO models

best_lasso_ds = lasso_cv_ds.best_estimator_.named_steps["lasso"]
best_lasso_gm = lasso_cv_gm.best_estimator_.named_steps["lasso"]

coef_ds = pd.Series(
    best_lasso_ds.coef_,
    index=deepseek_predictors
)

coef_gm = pd.Series(
    best_lasso_gm.coef_,
    index=gemma_predictors
)

selected_ds = (
    coef_ds[coef_ds != 0]
    .sort_values(key=np.abs, ascending=False)
)

selected_gm = (
    coef_gm[coef_gm != 0]
    .sort_values(key=np.abs, ascending=False)
)

print("DEEPSEEK — selected predictors")
display(selected_ds.to_frame("coefficient"))
print("Number selected:", len(selected_ds))

print("\nGEMMA — selected predictors")
display(selected_gm.to_frame("coefficient"))
print("Number selected:", len(selected_gm))

DEEPSEEK — selected predictors


,coefficient
VIXCLS,0.000125
VIXCLS_ma3,0.000042


Number selected: 2

GEMMA — selected predictors


,coefficient
VIXCLS,0.000076


Number selected: 1


In [11]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Leakage-safe LASSO predictions
pred_ds = lasso_cv_ds.predict(X_test_ds)
pred_gm = lasso_cv_gm.predict(X_test_gm)

# Naive benchmark:
# predict each test observation using the mean next-day absolute return
# observed in the TRAINING sample only
benchmark_ds = np.repeat(y_train_ds.mean(), len(y_test_ds))
benchmark_gm = np.repeat(y_train_gm.mean(), len(y_test_gm))

def evaluate(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame({
    "DeepSeek LASSO": evaluate(y_test_ds, pred_ds),
    "DeepSeek Benchmark": evaluate(y_test_ds, benchmark_ds),
    "Gemma LASSO": evaluate(y_test_gm, pred_gm),
    "Gemma Benchmark": evaluate(y_test_gm, benchmark_gm)
}).T

display(results)

print("\nRMSE improvement over naive benchmark:")

print(
    "DeepSeek:",
    100 * (
        results.loc["DeepSeek Benchmark", "RMSE"]
        - results.loc["DeepSeek LASSO", "RMSE"]
    ) / results.loc["DeepSeek Benchmark", "RMSE"],
    "%"
)

print(
    "Gemma:",
    100 * (
        results.loc["Gemma Benchmark", "RMSE"]
        - results.loc["Gemma LASSO", "RMSE"]
    ) / results.loc["Gemma Benchmark", "RMSE"],
    "%"
)

,RMSE,MAE,R2
DeepSeek LASSO,0.002623,0.002063,0.024234
DeepSeek Benchmark,0.002670,0.002028,-0.011142
Gemma LASSO,0.002641,0.002038,0.010069
Gemma Benchmark,0.002670,0.002028,-0.011142



RMSE improvement over naive benchmark:
DeepSeek: 1.7649245279771621 %
Gemma: 1.0544522140637322 %


In [12]:
print("Manifest shape:", manifest.shape)
print("\nManifest columns:")
print(manifest.columns.tolist())

display(manifest.head(10))

Manifest shape: (124, 3)

Manifest columns:
['feature', 'source', 'transformation']


,feature,source,transformation
0,DGS2,Macro,level_or_daily_aggregation
1,USEPUINDXD,Macro,level_or_daily_aggregation
2,VIXCLS,Macro,level_or_daily_aggregation
3,DGS2_diff,Macro,difference
4,DGS2_lag1,Macro,lag
5,DGS2_lag2,Macro,lag
6,DGS2_lag3,Macro,lag
7,DGS2_lag5,Macro,lag
8,DGS2_diff_lag1,Macro,lagged_difference
9,DGS2_diff_lag2,Macro,lagged_difference


In [13]:
# Define macro-only predictors from the feature manifest

macro_predictors = manifest.loc[
    manifest["source"].isin(["Macro", "EURUSD"]),
    "feature"
].tolist()

print("Macro-only predictor count:", len(macro_predictors))
print("\nFirst 15 macro-only predictors:")
print(macro_predictors[:15])

Macro-only predictor count: 43

First 15 macro-only predictors:
['DGS2', 'USEPUINDXD', 'VIXCLS', 'DGS2_diff', 'DGS2_lag1', 'DGS2_lag2', 'DGS2_lag3', 'DGS2_lag5', 'DGS2_diff_lag1', 'DGS2_diff_lag2', 'DGS2_diff_lag3', 'DGS2_diff_lag5', 'DGS2_ma3', 'DGS2_ma5', 'DGS2_ma10']


In [14]:
# Macro-only feature matrices
# Restrict to the same valid t -> t+1 modelling sample

X_macro_ds = (
    deepseek.loc[valid_ds, macro_predictors]
    .reset_index(drop=True)
    .copy()
)

X_macro_gm = (
    gemma.loc[valid_gm, macro_predictors]
    .reset_index(drop=True)
    .copy()
)

# Same chronological split as full models
X_macro_train_ds = X_macro_ds.iloc[:split_idx].copy()
X_macro_test_ds  = X_macro_ds.iloc[split_idx:].copy()

X_macro_train_gm = X_macro_gm.iloc[:split_idx].copy()
X_macro_test_gm  = X_macro_gm.iloc[split_idx:].copy()

print(
    "DeepSeek macro train/test:",
    X_macro_train_ds.shape,
    X_macro_test_ds.shape
)

print(
    "Gemma macro train/test:",
    X_macro_train_gm.shape,
    X_macro_test_gm.shape
)

print("\nMissing values:")
print("DeepSeek:", X_macro_ds.isna().sum().sum())
print("Gemma:", X_macro_gm.isna().sum().sum())

DeepSeek macro train/test: (784, 43) (197, 43)
Gemma macro train/test: (784, 43) (197, 43)

Missing values:
DeepSeek: 0
Gemma: 0


In [15]:
# Macro-only leakage-safe LASSO

macro_lasso_cv_ds = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

macro_lasso_cv_gm = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

macro_lasso_cv_ds.fit(X_macro_train_ds, y_train_ds)
macro_lasso_cv_gm.fit(X_macro_train_gm, y_train_gm)

print("DeepSeek macro-only best alpha:",
      macro_lasso_cv_ds.best_params_["lasso__alpha"])

print("Gemma macro-only best alpha:",
      macro_lasso_cv_gm.best_params_["lasso__alpha"])

DeepSeek macro-only best alpha: 0.0003180625692794119
Gemma macro-only best alpha: 0.0003180625692794119


In [16]:
# Extract selected macro-only LASSO predictors

best_macro_lasso_ds = macro_lasso_cv_ds.best_estimator_.named_steps["lasso"]
best_macro_lasso_gm = macro_lasso_cv_gm.best_estimator_.named_steps["lasso"]

coef_macro_ds = pd.Series(
    best_macro_lasso_ds.coef_,
    index=macro_predictors
)

coef_macro_gm = pd.Series(
    best_macro_lasso_gm.coef_,
    index=macro_predictors
)

selected_macro_ds = (
    coef_macro_ds[coef_macro_ds != 0]
    .sort_values(key=np.abs, ascending=False)
)

selected_macro_gm = (
    coef_macro_gm[coef_macro_gm != 0]
    .sort_values(key=np.abs, ascending=False)
)

print("DEEPSEEK — macro-only selected predictors:")
display(selected_macro_ds.to_frame("coefficient"))

print("\nGEMMA — macro-only selected predictors:")
display(selected_macro_gm.to_frame("coefficient"))

DEEPSEEK — macro-only selected predictors:


,coefficient
VIXCLS,0.000162
VIXCLS_ma3,0.000079



GEMMA — macro-only selected predictors:


,coefficient
VIXCLS,0.000162
VIXCLS_ma3,0.000079


In [17]:
# Compare naive benchmark, macro-only, DeepSeek and Gemma LASSO OOS performance

pred_macro_ds = macro_lasso_cv_ds.predict(X_macro_test_ds)
pred_macro_gm = macro_lasso_cv_gm.predict(X_macro_test_gm)

pred_full_ds = lasso_cv_ds.predict(X_test_ds)
pred_full_gm = lasso_cv_gm.predict(X_test_gm)

comparison = pd.DataFrame({
    "Naive Benchmark": evaluate(y_test_ds, benchmark_ds),
    "Macro-only": evaluate(y_test_ds, pred_macro_ds),
    "DeepSeek": evaluate(y_test_ds, pred_full_ds),
    "Gemma": evaluate(y_test_gm, pred_full_gm)
}).T

display(comparison)

# RMSE improvement relative to macro-only model
macro_rmse = comparison.loc["Macro-only", "RMSE"]

print("\nRMSE improvement relative to macro-only:")

print(
    "DeepSeek:",
    100 * (macro_rmse - comparison.loc["DeepSeek", "RMSE"]) / macro_rmse,
    "%"
)

print(
    "Gemma:",
    100 * (macro_rmse - comparison.loc["Gemma", "RMSE"]) / macro_rmse,
    "%"
)

,RMSE,MAE,R2
Naive Benchmark,0.002670,0.002028,-0.011142
Macro-only,0.002620,0.002095,0.026197
DeepSeek,0.002623,0.002063,0.024234
Gemma,0.002641,0.002038,0.010069



RMSE improvement relative to macro-only:
DeepSeek: -0.10070616204602176 %
Gemma: -0.8246714055152107 %


## LASSO Results — Next-Day Absolute EUR/USD Log Return

Using the one-day-ahead volatility proxy, the macro-only LASSO selects **VIXCLS** and **VIXCLS_ma3**. The full DeepSeek model selects the same two predictors, while the Gemma model retains only **VIXCLS**. No geopolitical variable is selected by LASSO.

Out-of-sample, the macro-only model achieves an RMSE of approximately **0.002620**. Adding the DeepSeek predictors produces an RMSE of approximately **0.002623**, around **0.10% worse** than macro-only, while the Gemma specification produces an RMSE of approximately **0.002641**, around **0.82% worse**.

The results therefore provide no evidence that the LLM-derived geopolitical indicators improve one-day-ahead volatility prediction under LASSO. The variables retained by the model are exclusively VIX-based, suggesting that conventional market uncertainty contains the more stable linear predictive signal at this horizon.

# Elastic Net — Absolute EUR/USD Log-Return Target

In [18]:
# Elastic Net setup

from sklearn.linear_model import ElasticNet

elastic_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic", ElasticNet(max_iter=100000))
])

# Same alpha range as LASSO
elastic_alphas = alphas

# l1_ratio controls the LASSO/Ridge mixture:
# 1.0 = LASSO
# closer to 0 = more Ridge-like
l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9]

elastic_param_grid = {
    "elastic__alpha": elastic_alphas,
    "elastic__l1_ratio": l1_ratios
}

print("Number of alpha values:", len(elastic_alphas))
print("Alpha range:", elastic_alphas.min(), "to", elastic_alphas.max())
print("L1 ratios:", l1_ratios)
print(
    "Total parameter combinations:",
    len(elastic_alphas) * len(l1_ratios)
)

Number of alpha values: 200
Alpha range: 1e-06 to 1.0
L1 ratios: [0.1, 0.25, 0.5, 0.75, 0.9]
Total parameter combinations: 1000


In [19]:
# Macro-only leakage-safe Elastic Net

macro_elastic_cv = GridSearchCV(
    estimator=elastic_pipe,
    param_grid=elastic_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

macro_elastic_cv.fit(X_macro_train_ds, y_train_ds)

print("Macro-only best alpha:",
      macro_elastic_cv.best_params_["elastic__alpha"])

print("Macro-only best l1_ratio:",
      macro_elastic_cv.best_params_["elastic__l1_ratio"])

Macro-only best alpha: 0.0012750512407130128
Macro-only best l1_ratio: 0.25


In [20]:
# Extract macro-only Elastic Net coefficients

best_macro_elastic = macro_elastic_cv.best_estimator_.named_steps["elastic"]

coef_macro_elastic = pd.Series(
    best_macro_elastic.coef_,
    index=macro_predictors
)

selected_macro_elastic = (
    coef_macro_elastic[coef_macro_elastic != 0]
    .sort_values(key=np.abs, ascending=False)
)

print("MACRO-ONLY — Elastic Net selected predictors:")
display(selected_macro_elastic.to_frame("coefficient"))

print("\nNumber selected:", len(selected_macro_elastic))

MACRO-ONLY — Elastic Net selected predictors:


,coefficient
VIXCLS,0.000160
VIXCLS_ma3,0.000081



Number selected: 2


In [21]:
# Full DeepSeek Elastic Net

elastic_cv_ds = GridSearchCV(
    estimator=elastic_pipe,
    param_grid=elastic_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

elastic_cv_ds.fit(X_train_ds, y_train_ds)

print("DeepSeek best alpha:",
      elastic_cv_ds.best_params_["elastic__alpha"])

print("DeepSeek best l1_ratio:",
      elastic_cv_ds.best_params_["elastic__l1_ratio"])


# Full Gemma Elastic Net

elastic_cv_gm = GridSearchCV(
    estimator=elastic_pipe,
    param_grid=elastic_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

elastic_cv_gm.fit(X_train_gm, y_train_gm)

print("\nGemma best alpha:",
      elastic_cv_gm.best_params_["elastic__alpha"])

print("Gemma best l1_ratio:",
      elastic_cv_gm.best_params_["elastic__l1_ratio"])

DeepSeek best alpha: 0.000419870708444391
DeepSeek best l1_ratio: 0.9

Gemma best alpha: 0.0005542664520663101
Gemma best l1_ratio: 0.9


In [22]:
# Extract full Elastic Net selected predictors

best_elastic_ds = elastic_cv_ds.best_estimator_.named_steps["elastic"]
best_elastic_gm = elastic_cv_gm.best_estimator_.named_steps["elastic"]

coef_elastic_ds = pd.Series(
    best_elastic_ds.coef_,
    index=X_train_ds.columns
)

coef_elastic_gm = pd.Series(
    best_elastic_gm.coef_,
    index=X_train_gm.columns
)

selected_elastic_ds = (
    coef_elastic_ds[coef_elastic_ds != 0]
    .sort_values(key=np.abs, ascending=False)
)

selected_elastic_gm = (
    coef_elastic_gm[coef_elastic_gm != 0]
    .sort_values(key=np.abs, ascending=False)
)

print("DEEPSEEK — Elastic Net selected predictors:")
display(selected_elastic_ds.to_frame("coefficient"))
print("Number selected:", len(selected_elastic_ds))

print("\nGEMMA — Elastic Net selected predictors:")
display(selected_elastic_gm.to_frame("coefficient"))
print("Number selected:", len(selected_elastic_gm))

DEEPSEEK — Elastic Net selected predictors:


,coefficient
VIXCLS,0.000132
VIXCLS_ma3,0.000049


Number selected: 2

GEMMA — Elastic Net selected predictors:


,coefficient
VIXCLS,0.000059


Number selected: 1


In [23]:
# Elastic Net OOS predictions

pred_macro_elastic = macro_elastic_cv.predict(X_macro_test_ds)
pred_elastic_ds = elastic_cv_ds.predict(X_test_ds)
pred_elastic_gm = elastic_cv_gm.predict(X_test_gm)

elastic_comparison = pd.DataFrame({
    "Naive Benchmark": evaluate(y_test_ds, benchmark_ds),
    "Macro-only": evaluate(y_test_ds, pred_macro_elastic),
    "DeepSeek": evaluate(y_test_ds, pred_elastic_ds),
    "Gemma": evaluate(y_test_gm, pred_elastic_gm)
}).T

display(elastic_comparison)

# RMSE improvement relative to macro-only
macro_elastic_rmse = elastic_comparison.loc["Macro-only", "RMSE"]

print("\nRMSE improvement relative to macro-only:")

print(
    "DeepSeek:",
    100 * (
        macro_elastic_rmse
        - elastic_comparison.loc["DeepSeek", "RMSE"]
    ) / macro_elastic_rmse,
    "%"
)

print(
    "Gemma:",
    100 * (
        macro_elastic_rmse
        - elastic_comparison.loc["Gemma", "RMSE"]
    ) / macro_elastic_rmse,
    "%"
)

,RMSE,MAE,R2
Naive Benchmark,0.002670,0.002028,-0.011142
Macro-only,0.002620,0.002095,0.026177
DeepSeek,0.002621,0.002069,0.025253
Gemma,0.002647,0.002035,0.006154



RMSE improvement relative to macro-only:
DeepSeek: -0.04739447454238794 %
Gemma: -1.0228103652178786 %


## Elastic Net Results — Next-Day Absolute EUR/USD Log Return

Elastic Net produces results that are very similar to LASSO.

The macro-only model retains **VIXCLS** and **VIXCLS_ma3**. The full DeepSeek model retains the same two predictors, while the Gemma model retains only **VIXCLS**. No geopolitical predictor survives the cross-validated Elastic Net penalty.

Out-of-sample, the macro-only model achieves an RMSE of approximately **0.002620**. The DeepSeek specification achieves an RMSE of approximately **0.002621**, around **0.05% worse** than macro-only, while the Gemma specification achieves an RMSE of approximately **0.002647**, around **1.02% worse**.

Elastic Net therefore reinforces the LASSO result. Allowing correlated predictors to enter jointly through a combination of L1 and L2 regularisation does not reveal incremental predictive value from the LLM-derived geopolitical indicators for next-day EUR/USD movement magnitude. The stable predictive signal remains concentrated in VIX-based variables.

# Ridge — Absolute EUR/USD Log-Return Target

In [24]:
# Ridge setup

from sklearn.linear_model import Ridge

ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# Wide penalty grid
ridge_alphas = np.logspace(-6, 8, 250)

ridge_param_grid = {
    "ridge__alpha": ridge_alphas
}

print("Number of alpha values:", len(ridge_alphas))
print("Alpha range:", ridge_alphas.min(), "to", ridge_alphas.max())

Number of alpha values: 250
Alpha range: 1e-06 to 100000000.0


In [25]:
# Macro-only leakage-safe Ridge

macro_ridge_cv = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

macro_ridge_cv.fit(X_macro_train_ds, y_train_ds)

print(
    "Macro-only best alpha:",
    macro_ridge_cv.best_params_["ridge__alpha"]
)

Macro-only best alpha: 100000000.0


In [26]:
# Inspect Ridge CV behaviour at large penalties

ridge_cv_results = pd.DataFrame(macro_ridge_cv.cv_results_)

ridge_cv_results["mean_mse"] = -ridge_cv_results["mean_test_score"]

ridge_summary = ridge_cv_results[
    ["param_ridge__alpha", "mean_mse"]
].copy()

ridge_summary["param_ridge__alpha"] = ridge_summary[
    "param_ridge__alpha"
].astype(float)

ridge_summary = ridge_summary.sort_values(
    "param_ridge__alpha"
)

display(ridge_summary.tail(15))

,param_ridge__alpha,mean_mse
235,1.632493e+07,0.000007
236,1.858131e+07,0.000007
237,2.114955e+07,0.000007
238,2.407277e+07,0.000007
239,2.740003e+07,0.000007
240,3.118717e+07,0.000007
241,3.549775e+07,0.000007
242,4.040412e+07,0.000007
243,4.598864e+07,0.000007
244,5.234504e+07,0.000007


In [27]:
# Full DeepSeek and Gemma leakage-safe Ridge

ridge_cv_ds = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

ridge_cv_gm = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

ridge_cv_ds.fit(X_train_ds, y_train_ds)

print(
    "DeepSeek best alpha:",
    ridge_cv_ds.best_params_["ridge__alpha"]
)

ridge_cv_gm.fit(X_train_gm, y_train_gm)

print(
    "Gemma best alpha:",
    ridge_cv_gm.best_params_["ridge__alpha"]
)

DeepSeek best alpha: 100000000.0
Gemma best alpha: 17097.465139375156


In [28]:
# Ridge OOS predictions

pred_macro_ridge = macro_ridge_cv.predict(X_macro_test_ds)
pred_ridge_ds = ridge_cv_ds.predict(X_test_ds)
pred_ridge_gm = ridge_cv_gm.predict(X_test_gm)

ridge_comparison = pd.DataFrame({
    "Naive Benchmark": evaluate(y_test_ds, benchmark_ds),
    "Macro-only": evaluate(y_test_ds, pred_macro_ridge),
    "DeepSeek": evaluate(y_test_ds, pred_ridge_ds),
    "Gemma": evaluate(y_test_gm, pred_ridge_gm)
}).T

display(ridge_comparison)

# RMSE improvement relative to macro-only
macro_ridge_rmse = ridge_comparison.loc["Macro-only", "RMSE"]

print("\nRMSE improvement relative to macro-only:")

print(
    "DeepSeek:",
    100 * (
        macro_ridge_rmse
        - ridge_comparison.loc["DeepSeek", "RMSE"]
    ) / macro_ridge_rmse,
    "%"
)

print(
    "Gemma:",
    100 * (
        macro_ridge_rmse
        - ridge_comparison.loc["Gemma", "RMSE"]
    ) / macro_ridge_rmse,
    "%"
)

,RMSE,MAE,R2
Naive Benchmark,0.002670,0.002028,-0.011142
Macro-only,0.002670,0.002028,-0.011108
DeepSeek,0.002670,0.002028,-0.011110
Gemma,0.002724,0.002282,-0.052658



RMSE improvement relative to macro-only:
DeepSeek: -7.289348486917271e-05 %
Gemma: -2.0339823930416947 %


In [29]:
# Inspect largest Ridge coefficients

ridge_coef_ds = pd.Series(
    ridge_cv_ds.best_estimator_.named_steps["ridge"].coef_,
    index=X_train_ds.columns
).sort_values(key=np.abs, ascending=False)

ridge_coef_gm = pd.Series(
    ridge_cv_gm.best_estimator_.named_steps["ridge"].coef_,
    index=X_train_gm.columns
).sort_values(key=np.abs, ascending=False)

print("DEEPSEEK — 20 largest Ridge coefficients:")
display(ridge_coef_ds.head(20).to_frame("coefficient"))

print("\nGEMMA — 20 largest Ridge coefficients:")
display(ridge_coef_gm.head(20).to_frame("coefficient"))

DEEPSEEK — 20 largest Ridge coefficients:


,coefficient
VIXCLS,4.373953e-09
VIXCLS_ma3,4.362282e-09
VIXCLS_lag2,4.254664e-09
VIXCLS_lag1,4.230154e-09
VIXCLS_ma5,4.170129e-09
VIXCLS_ma10,3.708284e-09
VIXCLS_lag3,3.534289e-09
VIXCLS_lag5,3.260981e-09
USEPUINDXD_ma5,3.198136e-09
USEPUINDXD_ma3,3.168584e-09



GEMMA — 20 largest Ridge coefficients:


,coefficient
VIXCLS,0.000018
VIXCLS_ma3,0.000018
VIXCLS_lag2,0.000018
VIXCLS_lag1,0.000017
VIXCLS_ma5,0.000017
VIXCLS_ma10,0.000015
VIXCLS_diff_lag2,0.000014
VIXCLS_lag3,0.000014
VIXCLS_lag5,0.000013
USEPUINDXD_ma5,0.000012


In [30]:
# Final data, target and leakage audit

print("=" * 65)
print("VOLATILITY-TARGET REGULARISATION — FINAL AUDIT")
print("=" * 65)

# 1. Dataset sizes
print("\n1. DATASET SIZES")
print("DeepSeek full X:", X_train_ds.shape, X_test_ds.shape)
print("Gemma full X:   ", X_train_gm.shape, X_test_gm.shape)
print("Macro-only X:   ", X_macro_train_ds.shape, X_macro_test_ds.shape)

# 2. Target sizes
print("\n2. TARGET SIZES")
print("DeepSeek y:", len(y_train_ds), len(y_test_ds))
print("Gemma y:   ", len(y_train_gm), len(y_test_gm))

# 3. DeepSeek / Gemma target equality
train_targets_equal = np.allclose(
    np.asarray(y_train_ds),
    np.asarray(y_train_gm),
    equal_nan=True
)

test_targets_equal = np.allclose(
    np.asarray(y_test_ds),
    np.asarray(y_test_gm),
    equal_nan=True
)

print("\n3. DEEPSEEK / GEMMA TARGET EQUALITY")
print("Training targets identical:", train_targets_equal)
print("Test targets identical:    ", test_targets_equal)

# 4. Target properties
# Absolute log return must be non-negative
print("\n4. TARGET PROPERTIES")
print("Target variable:", TARGET_VOL)
print("Expected target: absolute EUR/USD daily log return")

print(
    "DeepSeek target non-negative:",
    (np.asarray(y_train_ds) >= 0).all()
    and (np.asarray(y_test_ds) >= 0).all()
)

print(
    "Gemma target non-negative:",
    (np.asarray(y_train_gm) >= 0).all()
    and (np.asarray(y_test_gm) >= 0).all()
)

# 5. Leakage check
print("\n5. TARGET LEAKAGE CHECK")

for name, X in {
    "DeepSeek": X_train_ds,
    "Gemma": X_train_gm,
    "Macro-only": X_macro_train_ds
}.items():

    print(
        f"{name}:",
        f"{TARGET_RETURN} in X =", TARGET_RETURN in X.columns,
        "|",
        f"{TARGET_VOL} in X =", TARGET_VOL in X.columns
    )

# 6. Missing values
print("\n6. MISSING VALUES")

missing_ds = (
    X_train_ds.isna().sum().sum()
    + X_test_ds.isna().sum().sum()
)

missing_gm = (
    X_train_gm.isna().sum().sum()
    + X_test_gm.isna().sum().sum()
)

missing_macro = (
    X_macro_train_ds.isna().sum().sum()
    + X_macro_test_ds.isna().sum().sum()
)

missing_y_ds = (
    pd.isna(y_train_ds).sum()
    + pd.isna(y_test_ds).sum()
)

missing_y_gm = (
    pd.isna(y_train_gm).sum()
    + pd.isna(y_test_gm).sum()
)

print("DeepSeek X:", missing_ds)
print("Gemma X:   ", missing_gm)
print("Macro X:   ", missing_macro)
print("DeepSeek y:", missing_y_ds)
print("Gemma y:   ", missing_y_gm)

# 7. Predictor counts
print("\n7. PREDICTOR COUNTS")
print("DeepSeek:", X_train_ds.shape[1])
print("Gemma:   ", X_train_gm.shape[1])
print("Macro:   ", X_macro_train_ds.shape[1])

# 8. Overall verdict
no_target_leakage = (
    TARGET_RETURN not in X_train_ds.columns
    and TARGET_VOL not in X_train_ds.columns
    and TARGET_RETURN not in X_train_gm.columns
    and TARGET_VOL not in X_train_gm.columns
    and TARGET_RETURN not in X_macro_train_ds.columns
    and TARGET_VOL not in X_macro_train_ds.columns
)

targets_nonnegative = (
    (np.asarray(y_train_ds) >= 0).all()
    and (np.asarray(y_test_ds) >= 0).all()
    and (np.asarray(y_train_gm) >= 0).all()
    and (np.asarray(y_test_gm) >= 0).all()
)

all_checks = (
    train_targets_equal
    and test_targets_equal
    and targets_nonnegative
    and no_target_leakage
    and missing_ds == 0
    and missing_gm == 0
    and missing_macro == 0
    and missing_y_ds == 0
    and missing_y_gm == 0
    and X_train_ds.shape[1] == 124
    and X_train_gm.shape[1] == 124
    and X_macro_train_ds.shape[1] == 43
)

print("\n" + "=" * 65)

if all_checks:
    print("AUDIT PASSED")
else:
    print("AUDIT FAILED — inspect checks above")

print("=" * 65)

VOLATILITY-TARGET REGULARISATION — FINAL AUDIT

1. DATASET SIZES
DeepSeek full X: (784, 124) (197, 124)
Gemma full X:    (784, 124) (197, 124)
Macro-only X:    (784, 43) (197, 43)

2. TARGET SIZES
DeepSeek y: 784 197
Gemma y:    784 197

3. DEEPSEEK / GEMMA TARGET EQUALITY
Training targets identical: True
Test targets identical:     True

4. TARGET PROPERTIES
Target variable: DEXUSEU_abs_logreturn_tplus1
Expected target: absolute EUR/USD daily log return
DeepSeek target non-negative: True
Gemma target non-negative: True

5. TARGET LEAKAGE CHECK
DeepSeek: DEXUSEU_logreturn in X = False | DEXUSEU_abs_logreturn_tplus1 in X = False
Gemma: DEXUSEU_logreturn in X = False | DEXUSEU_abs_logreturn_tplus1 in X = False
Macro-only: DEXUSEU_logreturn in X = False | DEXUSEU_abs_logreturn_tplus1 in X = False

6. MISSING VALUES
DeepSeek X: 0
Gemma X:    0
Macro X:    0
DeepSeek y: 0
Gemma y:    0

7. PREDICTOR COUNTS
DeepSeek: 124
Gemma:    124
Macro:    43

AUDIT PASSED
